In [11]:
import importlib
import sys

In [12]:
for key in list(sys.modules.keys()):
    if 'collect' in key:
        del sys.modules[key]

In [20]:
import requests
import pandas as pd
import time
import os
import xml.etree.ElementTree as ET

os.makedirs("E:/Research Paper topic/data", exist_ok=True)

def fetch_arxiv_papers(query, total=300):
    papers = []
    batch_size = 100
    start = 0
    query_formatted = query.replace(" ", "+AND+")

    while len(papers) < total:
        url = (
            "http://export.arxiv.org/api/query?"
            "search_query=all:" + query_formatted +
            "&start=" + str(start) +
            "&max_results=" + str(batch_size) +
            "&sortBy=submittedDate&sortOrder=descending"
        )
        response = requests.get(url)
        root = ET.fromstring(response.content)
        ns = {'atom': 'http://www.w3.org/2005/Atom'}
        entries = root.findall('atom:entry', ns)

        if not entries:
            print("No more results found.")
            break

        for entry in entries:
            title      = entry.find('atom:title', ns)
            abstract   = entry.find('atom:summary', ns)
            published  = entry.find('atom:published', ns)
            authors    = entry.findall('atom:author', ns)
            categories = entry.findall('atom:category', ns)

            papers.append({
                "title":    title.text.strip().replace("\n", " ") if title is not None else "",
                "abstract": abstract.text.strip().replace("\n", " ") if abstract is not None else "",
                "year":     published.text[:4] if published is not None else "",
                "authors":  ", ".join([
                    a.find('atom:name', ns).text
                    for a in authors
                    if a.find('atom:name', ns) is not None
                ]),
                "venue": "arXiv",
                "citations": 0,
                "category": categories[0].get('term', '') if categories else ""
            })

        print("Fetched " + str(len(papers)) + " papers so far...")
        start += batch_size
        time.sleep(3)

    save_path = "E:/Research Paper topic/data/raw_papers.csv"
    df = pd.DataFrame(papers[:total])
    df.to_csv(save_path, index=False)
    print("Done! Saved " + str(len(df)) + " papers to " + save_path)
    return df

print("Function defined successfully!")

Function defined successfully!


In [21]:
df = fetch_arxiv_papers("reinforcement learning robotics", total=300)
df.head()

Fetched 100 papers so far...
Fetched 200 papers so far...
Fetched 300 papers so far...
Done! Saved 300 papers to E:/Research Paper topic/data/raw_papers.csv


,title,abstract,year,authors,venue,citations,category
0,BORA: Bridging Offline Reinforcement Learning ...,Vision-Language-Action (VLA) models have emerg...,2026,"Zhongxi Chen, Yifan Han, Yanming Shao, Huanmin...",arXiv,0,cs.RO
1,Sample-Efficient Diffusion-based Reinforcement...,Recent advances in reinforcement learning (RL)...,2026,"Shutong Ding, Zejia Zhong, Zhongyi Wang, Ke Hu...",arXiv,0,cs.RO
2,LLM-Guided Future Hypotheses for Horizon-Aware...,Multi-step robot manipulation requires acting ...,2026,"Mohammad Khoshnazar, Andrew Melnik, Michael Beetz",arXiv,0,cs.RO
3,Momentum Based Reward Design for Low Emission ...,Urban traffic congestion is a growing global i...,2026,"Chinmay Mundane, Amith Manoharan, Arun Singh",arXiv,0,cs.LG
4,VE2VF: Vision-Enabled to Vision-Free Distillat...,When using reinforcement learning (RL) for con...,2026,"Victor Kowalski, Chengxi Li, Dongheui Lee",arXiv,0,cs.RO
